In [1]:
import time
from pathlib import Path
from PIL import Image
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

device = torch.device("cpu")
print(f"Active compute device: {device}")

Active compute device: cpu


In [2]:
split_csv_path = Path(r"C:\Users\HP\Desktop\projects\plantdiseaseprediction\data\dataset_splits.csv")
df = pd.read_csv(split_csv_path)

# Extract and sort 38 class names to build label encoding
classes = sorted(df["class_name"].unique())
class_to_idx = {name: idx for idx, name in enumerate(classes)}

class PlantVillageDataset(Dataset):
    def __init__(self, data_frame, transform=None):
        self.data = data_frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        label = class_to_idx[row["class_name"]]

        if self.transform:
            image = self.transform(image)

        return image, label

print(f"Total dataset entries: {len(df)} across {len(classes)} classes.")

Total dataset entries: 54305 across 38 classes.


In [7]:
from sklearn.model_selection import train_test_split

# 1. Pull the train and validation sets from df
raw_train_df = df[df["split"] == "train"].reset_index(drop=True)
raw_val_df = df[df["split"] == "val"].reset_index(drop=True)

# 2. Stratified 10% sampling using scikit-learn (fast and bug-free)
train_df, _ = train_test_split(
    raw_train_df,
    train_size=0.10,
    stratify=raw_train_df["class_name"],
    random_state=42
)

val_df, _ = train_test_split(
    raw_val_df,
    train_size=0.10,
    stratify=raw_val_df["class_name"],
    random_state=42
)

print(f"Subsampled data -> Train: {len(train_df)} images | Val: {len(val_df)} images")

# 3. Optimized 128x128 image transformations for CPU
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 4. Create datasets and loaders
train_dataset = PlantVillageDataset(train_df, transform=train_transform)
val_dataset = PlantVillageDataset(val_df, transform=eval_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Batches per epoch -> Train: {len(train_loader)} | Val: {len(val_loader)}")

Subsampled data -> Train: 3801 images | Val: 814 images
Batches per epoch -> Train: 119 | Val: 26


In [8]:
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all convolutional feature layers to minimize CPU backpropagation
for param in model.parameters():
    param.requires_grad = False

# Replace final classification head (38 output nodes)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(classes))

model = model.to(device)

criterion = nn.CrossEntropyLoss()
# Optimize strictly the parameters in the final layer
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

print("Classification head successfully adapted for CPU training:")
print(model.fc)

Classification head successfully adapted for CPU training:
Linear(in_features=512, out_features=38, bias=True)


In [9]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    total_batches = len(dataloader)

    for batch_idx, (images, labels) in enumerate(dataloader, 1):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        # Print progress every 30 batches
        if batch_idx % 30 == 0 or batch_idx == total_batches:
            print(f"  Batch {batch_idx:3d}/{total_batches} | Running Acc: {correct/total:.4f} | Loss: {running_loss/total:.4f}")

    return running_loss / total, correct / total


def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


# Run 2 baseline epochs (should complete in ~2-4 minutes total)
epochs = 2
print(f"Beginning CPU baseline training for {epochs} epochs...\n")

for epoch in range(1, epochs + 1):
    epoch_start = time.time()
    
    print(f"--- Epoch {epoch}/{epochs} ---")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    
    duration = time.time() - epoch_start
    print(f"Completed Epoch {epoch} in {duration:.1f}s")
    print(f"Train Acc: {train_acc * 100:.2f}% | Train Loss: {train_loss:.4f}")
    print(f"Val Acc:   {val_acc * 100:.2f}% | Val Loss:   {val_loss:.4f}\n")

Beginning CPU baseline training for 2 epochs...

--- Epoch 1/2 ---
  Batch  30/119 | Running Acc: 0.2146 | Loss: 3.0871
  Batch  60/119 | Running Acc: 0.3755 | Loss: 2.5503
  Batch  90/119 | Running Acc: 0.4736 | Loss: 2.1785
  Batch 119/119 | Running Acc: 0.5333 | Loss: 1.9509
Completed Epoch 1 in 109.0s
Train Acc: 53.33% | Train Loss: 1.9509
Val Acc:   75.06% | Val Loss:   1.0576

--- Epoch 2/2 ---
  Batch  30/119 | Running Acc: 0.7990 | Loss: 0.9193
  Batch  60/119 | Running Acc: 0.8063 | Loss: 0.8737
  Batch  90/119 | Running Acc: 0.8115 | Loss: 0.8371
  Batch 119/119 | Running Acc: 0.8179 | Loss: 0.8061
Completed Epoch 2 in 66.0s
Train Acc: 81.79% | Train Loss: 0.8061
Val Acc:   81.82% | Val Loss:   0.6923



In [6]:
print(df.columns.tolist())

['filepath', 'filename', 'class_name', 'split']


In [10]:
# Create models directory if it doesn't exist
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# 1. Save trained baseline weights
model_save_path = models_dir / "resnet18_baseline_cpu.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to: {model_save_path.resolve()}")

# 2. Evaluate on the Test Set
test_df_sub, _ = train_test_split(
    df[df["split"] == "test"].reset_index(drop=True),
    train_size=0.10,
    stratify=df[df["split"] == "test"]["class_name"],
    random_state=42
)
test_dataset = PlantVillageDataset(test_df_sub, transform=eval_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"\nFinal Baseline Test Acc:  {test_acc * 100:.2f}% | Test Loss: {test_loss:.4f}")

Model saved to: C:\Users\HP\Desktop\projects\plantdiseaseprediction\PlantDiseaseProject\models\resnet18_baseline_cpu.pth

Final Baseline Test Acc:  83.42% | Test Loss: 0.6951


In [11]:
from pathlib import Path
import torch

# Explicit absolute path to your project root's models directory
models_dir = Path(r"C:\Users\HP\Desktop\projects\plantdiseaseprediction\models")
models_dir.mkdir(parents=True, exist_ok=True)

save_target = models_dir / "resnet18_baseline_cpu.pth"
torch.save(model.state_dict(), save_target)

print(f"File exists: {save_target.exists()}")
print(f"Saved to: {save_target.resolve()}")

File exists: True
Saved to: C:\Users\HP\Desktop\projects\plantdiseaseprediction\models\resnet18_baseline_cpu.pth
